# Анализ оптовых продаж аудиотехники

Компания «Карпов Саунд» — оптовый поставщик аудиотехники: профессиональные аудиосистемы, домашние кинотеатры,
портативная акустика и аксессуары. Клиенты оставляют заявки в CRM, менеджеры их обрабатывают, после чего заказ
либо подтверждается (`confirmed`), либо отменяется (`canceled`).

Хранилище данных временно недоступно, поэтому данные за март 2024 года есть только в виде резервной выгрузки,
разложенной по папкам. Нужно собрать их, проанализировать и сформулировать выводы для руководства.

**План анализа**

1. [Сбор данных](#1.-Сбор-данных)
2. [Динамика заказов и аномальные дни](#2.-Динамика-заказов-и-аномальные-дни)
3. [Ключевые метрики месяца](#3.-Ключевые-метрики-месяца)
4. [Спрос на бренды](#4.-Спрос-на-бренды)
5. [Отчёт по менеджерам](#5.-Отчёт-по-менеджерам)
6. [Итоги](#Итоги)

## Данные

| Таблица | Поля |
|---|---|
| `orders` | `order_id` — номер заказа, `product_id` — идентификатор товара, `quantity` — количество товара в заказе |
| `order_status` | `order_id` — номер заказа, `client_id` — идентификатор клиента, `status` — статус заказа |
| `products` | `id` — идентификатор товара, `name` — «бренд, модель», `price` — цена за единицу, USD |
| `usd_rate.txt` | курс доллара ЦБ на каждый день периода: `дата,курс,валюта` |

```
data/
├── orders/<дата>/<менеджер>/orders.csv, order_status.csv
├── products/<категория>/products.csv
└── usd_rate.txt
```

В одном заказе может быть несколько товаров. Если заказ отменили и создали такой же заново, в базе останутся
два заказа с разными номерами и статусами.

> Исходные данные предоставлены образовательной платформой и в репозиторий не входят —
> см. [`data/README.md`](../data/README.md).

## Настройка окружения

In [1]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
from matplotlib.patches import Patch
import seaborn as sns
from IPython.display import display

DATA_DIR = os.path.join('..', 'data')
OUTPUT_DIR = os.path.join(DATA_DIR, 'processed')
IMAGES_DIR = os.path.join('..', 'images')
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(IMAGES_DIR, exist_ok=True)

sns.set_theme(style='whitegrid', font_scale=1.05)
plt.rcParams.update({
    'figure.dpi': 110,
    'axes.titlesize': 14,
    'axes.titleweight': 'bold',
    'axes.titlelocation': 'left',
    'axes.spines.top': False,
    'axes.spines.right': False,
})
pd.set_option('display.float_format', lambda x: f'{x:,.2f}'.replace(',', ' '))

MAIN = '#2E86AB'
ACCENT = '#E4572E'
MUTED = '#C9CED6'
STATUS_COLORS = {'confirmed': MAIN, 'canceled': ACCENT}
STATUS_RU = {'confirmed': 'Подтверждён', 'canceled': 'Отменён'}
WEEKDAYS_RU = ['Пн', 'Вт', 'Ср', 'Чт', 'Пт', 'Сб', 'Вс']


def save_fig(fig, name):
    """Сохраняет график в папку images (используется в README)."""
    fig.savefig(os.path.join(IMAGES_DIR, name), bbox_inches='tight', facecolor='white')


def fmt_rub(value):
    """Форматирует сумму в рублях с разделителями разрядов."""
    return f'{value:,.2f} ₽'.replace(',', ' ')


def millions(x, pos=None):
    text = f'{x / 1e6:,.1f}'.replace(',', ' ').rstrip('0').rstrip('.')
    return f'{text} млн'


def format_date_axis(ax, dates):
    ax.xaxis.set_major_locator(mdates.DayLocator(interval=2))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%d.%m'))
    ax.set_xlim(dates.min() - pd.Timedelta(hours=18), dates.max() + pd.Timedelta(hours=18))
    ax.grid(axis='x', visible=False)

Matplotlib is building the font cache; this may take a moment.


## 1. Сбор данных

Обходим папки с помощью `os.walk()` и собираем три датафрейма:

- `df_orders` — заказы + колонки `manager` и `date` из пути к файлу;
- `df_order_status` — статусы заказов;
- `df_products` — товары + колонка `category` из пути к файлу.

Собранные таблицы сохраняются в `data/processed/`.

In [2]:
target_files = {'orders.csv', 'order_status.csv', 'products.csv'}
data_files = [
    os.path.join(path, file)
    for path, _, files in os.walk(DATA_DIR)
    for file in files
    if file in target_files
]

orders_list, order_status_list, products_list = [], [], []

for file_path in data_files:
    parts = os.path.normpath(file_path).split(os.sep)
    file_name = parts[-1]
    data = pd.read_csv(file_path)

    if file_name == 'orders.csv':
        data['manager'] = parts[-2]
        data['date'] = parts[-3]
        orders_list.append(data)
    elif file_name == 'order_status.csv':
        order_status_list.append(data)
    elif file_name == 'products.csv':
        data['category'] = parts[-2]
        products_list.append(data)

df_orders = (pd.concat(orders_list, ignore_index=True)
             .sort_values(['order_id', 'product_id'], ignore_index=True))
df_order_status = (pd.concat(order_status_list, ignore_index=True)
                   .sort_values('order_id', ignore_index=True))
df_products = (pd.concat(products_list, ignore_index=True)
               .sort_values('id', ignore_index=True))

df_orders.to_csv(os.path.join(OUTPUT_DIR, 'df_orders.csv'), index=False)
df_order_status.to_csv(os.path.join(OUTPUT_DIR, 'df_order_status.csv'), index=False)
df_products.to_csv(os.path.join(OUTPUT_DIR, 'df_products.csv'), index=False)

pd.DataFrame(
    {'строк': [len(df_orders), len(df_order_status), len(df_products)],
     'колонок': [df_orders.shape[1], df_order_status.shape[1], df_products.shape[1]]},
    index=['df_orders', 'df_order_status', 'df_products'],
)

ValueError: No objects to concatenate

Ожидаемые размеры: `df_orders` — (4603, 5), `df_order_status` — (346, 3), `df_products` — (1677, 4).

In [ ]:
df_orders['date'] = pd.to_datetime(df_orders['date'])
display(df_orders.head())
display(df_order_status.head())
display(df_products.head())

## 2. Динамика заказов и аномальные дни

### 2.1. Число заказов по дням

Считаем уникальные заказы за каждый день (для справки — и объём товаров в штуках).

In [ ]:
orders_by_day = (df_orders.groupby('date', as_index=False)
                 .agg(orders=('order_id', 'nunique'), items=('quantity', 'sum')))
orders_by_day['weekday'] = orders_by_day['date'].dt.dayofweek
orders_by_day['is_weekend'] = orders_by_day['weekday'] >= 5

peak_day = orders_by_day.loc[orders_by_day['orders'].idxmax(), 'date']
print(f'День с наибольшим числом заказов: {peak_day:%Y-%m-%d}')

orders_by_day.sort_values('orders', ascending=False).head()

### 2.2. Сезонность и выбросы

In [ ]:
holiday = pd.Timestamp('2024-03-08')
by_weekday = orders_by_day.groupby('weekday')['orders'].mean().reindex(range(7))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(17, 5.5), gridspec_kw={'width_ratios': [3, 1]})

ax1.bar(orders_by_day['date'], orders_by_day['orders'],
        color=np.where(orders_by_day['is_weekend'], MUTED, MAIN), width=0.8)
format_date_axis(ax1, orders_by_day['date'])
ax1.set(title='Число заказов по дням', xlabel='', ylabel='Заказов')
ax1.legend(handles=[Patch(color=MAIN, label='Будни'), Patch(color=MUTED, label='Выходные')],
           loc='upper right', frameon=False)

for day, label in [(peak_day, 'пик'), (holiday, '8 Марта')]:
    value = orders_by_day.loc[orders_by_day['date'] == day, 'orders']
    if not value.empty:
        ax1.annotate(label, xy=(day, value.iloc[0]), xytext=(0, 18), textcoords='offset points',
                     ha='center', color=ACCENT, fontweight='bold',
                     arrowprops=dict(arrowstyle='->', color=ACCENT))

ax2.bar(WEEKDAYS_RU, by_weekday.values, color=[MAIN] * 5 + [MUTED] * 2)
ax2.grid(axis='x', visible=False)
ax1.set_ylim(top=orders_by_day['orders'].max() * 1.15)
ax2.set(title='В среднем по дням недели', ylabel='Заказов в день')

fig.tight_layout()
save_fig(fig, 'orders_by_weekday.png')
plt.show()

Будние дни с наименьшим числом заказов:

In [ ]:
orders_by_day[~orders_by_day['is_weekend']].nsmallest(3, 'orders')

**Выводы**

- В заказах чётко видна недельная сезонность: по выходным заказов почти нет, основной поток приходится на будни.
- Пик заказов — **14 марта** (четверг), он заметно выше обычного будничного уровня.
- **8 марта** — пятница, но заказов почти нет: это праздничный выходной день.

### 2.3. Статусы заказов

Объединяем заказы со статусами в общий датафрейм `orders_status`.

In [ ]:
orders_status = df_orders.merge(df_order_status, on='order_id', how='left', validate='many_to_one')
assert orders_status['status'].notna().all(), 'Есть заказы без статуса'
assert len(orders_status) == len(df_orders)

orders_status.to_csv(os.path.join(OUTPUT_DIR, 'orders_status.csv'), index=False)
orders_status.head()

In [ ]:
status_share = (orders_status.drop_duplicates('order_id')['status']
                .value_counts()
                .to_frame('orders'))
status_share['share'] = (status_share['orders'] / status_share['orders'].sum()).round(2)
status_share

### 2.4. Статусы заказов по дням

In [ ]:
status_by_day = (orders_status.groupby(['date', 'status'])['order_id'].nunique()
                 .unstack(fill_value=0)
                 .reindex(columns=['confirmed', 'canceled'], fill_value=0)
                 .astype(int))

print('Дни без подтверждённых заказов:')
status_by_day[status_by_day['confirmed'] == 0]

In [ ]:
fig, ax = plt.subplots(figsize=(17, 5.5))

ax.bar(status_by_day.index, status_by_day['confirmed'], color=STATUS_COLORS['confirmed'],
       label=STATUS_RU['confirmed'], width=0.8)
ax.bar(status_by_day.index, status_by_day['canceled'], bottom=status_by_day['confirmed'],
       color=STATUS_COLORS['canceled'], label=STATUS_RU['canceled'], width=0.8)

prev_day = peak_day - pd.Timedelta(days=1)
top = status_by_day.sum(axis=1).max()
ax.set_ylim(top=top * 1.2)
ax.axvspan(prev_day - pd.Timedelta(hours=12), peak_day + pd.Timedelta(hours=12),
           color=ACCENT, alpha=0.08, zorder=0)
ax.text(prev_day + pd.Timedelta(hours=12), top * 1.17,
        f'{prev_day:%d.%m} – {peak_day:%d.%m}', ha='center', va='top', color=ACCENT, fontweight='bold')

format_date_axis(ax, status_by_day.index)
ax.set(title='Заказы по дням в разбивке по статусу', xlabel='', ylabel='Заказов')
ax.legend(frameon=False, loc='upper left')

fig.tight_layout()
save_fig(fig, 'orders_by_status.png')
plt.show()

**Выводы**

- 9 марта (суббота) — единственный день без подтверждённых заказов.
- Накануне пика, 13 марта, большая часть заказов была отменена, а в сам пиковый день, 14 марта, отмен нет,
  зато подтверждённых заказов рекордно много.

### 2.5. Что произошло 13–14 марта?

Две гипотезы:

1. **Неудачный день + компенсация.** 13 марта клиенты сами отменили заказы, а 14 марта менеджеры перевыполнили план
   за счёт *новых* заказов.
2. **Сбой CRM.** 13 марта заказы не удалось подтвердить, они автоматически отменились, и 14 марта клиенты оформили их
   *повторно*.

Отличить их можно по повторным заказам. Будем считать, что заказ 14 марта повторяет отменённый 13 марта,
если у них совпадают клиент, менеджер, число уникальных товаров и общее количество товаров в штуках
(при разных номерах заказа).

In [ ]:
order_profile = (orders_status
                 .groupby(['order_id', 'date', 'client_id', 'manager', 'status'], as_index=False)
                 .agg(unique_products=('product_id', 'nunique'), total_quantity=('quantity', 'sum')))

match_keys = ['client_id', 'manager', 'unique_products', 'total_quantity']
prev_canceled = order_profile.query('date == @prev_day and status == "canceled"')
peak_orders = order_profile.query('date == @peak_day')

repeated = peak_orders.merge(prev_canceled[match_keys + ['order_id']],
                             on=match_keys, suffixes=('', '_prev'))

n_peak = peak_orders['order_id'].nunique()
n_prev_canceled = prev_canceled['order_id'].nunique()
n_repeated = repeated['order_id'].nunique()

pd.DataFrame({
    'значение': [
        n_prev_canceled,
        n_peak,
        n_repeated,
        f'{n_repeated / n_peak:.0%}' if n_peak else '—',
        f'{repeated["order_id_prev"].nunique() / n_prev_canceled:.0%}' if n_prev_canceled else '—',
    ]
}, index=[
    f'Отменённых заказов {prev_day:%d.%m}',
    f'Всех заказов {peak_day:%d.%m}',
    f'Из них повторяют отменённые накануне',
    f'Доля повторов среди заказов {peak_day:%d.%m}',
    f'Доля отменённых {prev_day:%d.%m}, оформленных заново',
])

**Вывод:** значительная часть заказов 14 марта полностью повторяет отменённые накануне
(тот же клиент, менеджер и состав заказа). Это подтверждает гипотезу о **сбое CRM**: всплеск 14 марта —
технический эффект, а не результат акции или работы отдела продаж. При оценке динамики продаж эти два дня
стоит рассматривать вместе.

## 3. Ключевые метрики месяца

### 3.1. Курс доллара

Товары закупаются в долларах, а продаются в рублях по курсу ЦБ на дату продажи.

In [ ]:
currency = pd.read_csv(os.path.join(DATA_DIR, 'usd_rate.txt'), header=None,
                       names=['date', 'currency_rate', 'currency'])
currency = currency.drop(columns='currency')
currency['date'] = pd.to_datetime(currency['date'])

print(f'Средний курс доллара за месяц: {currency["currency_rate"].mean():.2f} ₽')
currency.head()

### 3.2. Выручка

Собираем общий датафрейм `df_full`: заказы + статусы + товары + курс. Выручка по позиции =
цена в долларах × курс на дату заказа × количество. Для метрик берём только подтверждённые заказы (`df_confirmed`).

In [ ]:
df_full = (orders_status
           .merge(df_products.drop(columns='category'), left_on='product_id', right_on='id',
                  how='left', validate='many_to_one')
           .drop(columns='id')
           .merge(currency, on='date', how='left', validate='many_to_one'))

assert df_full[['price', 'currency_rate']].notna().all().all(), 'Не для всех строк нашлись цена или курс'

df_full['price_rub'] = df_full['price'] * df_full['currency_rate']
df_full['revenue'] = df_full['price_rub'] * df_full['quantity']
df_full['brand'] = df_full['name'].str.split(',').str[0].str.strip()

df_confirmed = df_full.query('status == "confirmed"')

total_revenue = df_confirmed['revenue'].sum()
total_orders = df_confirmed['order_id'].nunique()
avg_check = total_revenue / total_orders

pd.DataFrame(
    {'значение': [fmt_rub(total_revenue), total_orders, fmt_rub(avg_check)]},
    index=['Выручка за месяц', 'Подтверждённых заказов', 'Средний чек'],
)

### 3.3. Динамика метрик внутри месяца

In [ ]:
all_days = pd.date_range(df_orders['date'].min(), df_orders['date'].max(), name='date')
daily = (df_confirmed.groupby('date')
         .agg(revenue=('revenue', 'sum'), orders=('order_id', 'nunique'))
         .reindex(all_days, fill_value=0)
         .reset_index())
# в дни без подтверждённых заказов средний чек не определён
daily['avg_check'] = daily['revenue'] / daily['orders'].replace(0, np.nan)

panels = [
    ('revenue', 'Выручка, ₽', MAIN, millions),
    ('orders', 'Подтверждённые заказы', '#3BB273', None),
    ('avg_check', 'Средний чек, ₽', ACCENT, millions),
]

fig, axes = plt.subplots(3, 1, figsize=(15, 11), sharex=True)
for ax, (col, title, color, formatter) in zip(axes, panels):
    ax.plot(daily['date'], daily[col], color=color, marker='o', markersize=4, linewidth=2)
    ax.axvline(peak_day, color='grey', linestyle='--', linewidth=1)
    ax.set(title=title, ylabel='')
    if formatter:
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(formatter))

format_date_axis(axes[-1], daily['date'])
for ax in axes[:-1]:
    ax.grid(axis='x', visible=False)
axes[0].text(peak_day, axes[0].get_ylim()[1], f' {peak_day:%d.%m}', color='grey', va='top')

fig.tight_layout()
save_fig(fig, 'daily_metrics.png')
plt.show()

**Выводы**

- Выручка и число заказов в течение месяца то растут, то падают; пик обоих показателей приходится на 14 марта.
- Выручка и число заказов не всегда меняются в одном направлении: крупные заказы могут дать больше выручки при меньшем числе заказов.
- Средний чек колеблется независимо от выручки и в пиковый день не максимален.

## 4. Спрос на бренды

### 4.1. Интерес клиентов к брендам

Статус здесь не важен: если товар бренда попал в заказ, клиент им интересовался.

In [ ]:
print(f'Брендов в заказах: {df_full["brand"].nunique()}')

### 4.2. Выручка по брендам

Выручку считаем только по подтверждённым заказам.

In [ ]:
brands = (df_confirmed.groupby('brand', as_index=False)
          .agg(revenue=('revenue', 'sum'), orders=('order_id', 'nunique'))
          .sort_values('revenue', ascending=False, ignore_index=True))
brands['revenue_share'] = (brands['revenue'] / brands['revenue'].sum() * 100).round(1)
brands['avg_revenue_per_order'] = brands['revenue'] / brands['orders']

top = brands.head(10)
fig, ax = plt.subplots(figsize=(12, 5.5))
sns.barplot(data=top, y='brand', x='revenue', color=MAIN, saturation=1, ax=ax)
ax.grid(axis='y', visible=False)
ax.set_xlim(right=top['revenue'].max() * 1.12)
ax.bar_label(ax.containers[0], labels=[f'{v:.1f}%' for v in top['revenue_share']], padding=4)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(millions))
ax.set(title='Топ-10 брендов по выручке', xlabel='Выручка, ₽', ylabel='')
fig.tight_layout()
save_fig(fig, 'brands_revenue.png')
plt.show()

print('Топ-5 по числу заказов:')
brands.sort_values('orders', ascending=False).head()

**Вывод:** лидер по выручке — **JBL**. Сравнение с топом по числу заказов показывает, опирается ли лидерство
бренда на частоту заказов или на крупные суммы.

### 4.3. Товары, которые ни разу не заказывали

In [ ]:
products_in_orders = df_confirmed['product_id'].unique()
df_products['is_in_orders'] = np.where(df_products['id'].isin(products_in_orders), 'yes', 'no')
df_products['brand'] = df_products['name'].str.split(',').str[0].str.strip()

df_products['is_in_orders'].value_counts()

### 4.4. Бренды, которые «захламляют полку»

Ищем бренды, у которых больше половины товаров ни разу не заказывали. Бренды с небольшим ассортиментом
(меньше 15 товаров) не учитываем — они не перегружают каталог.

In [ ]:
brand_stock = (df_products.groupby(['brand', 'is_in_orders'])['id'].nunique()
               .unstack(fill_value=0)
               .reindex(columns=['yes', 'no'], fill_value=0))
brand_stock['products'] = brand_stock['yes'] + brand_stock['no']
brand_stock['unsold_share'] = brand_stock['no'] / brand_stock['products']

big_brands = brand_stock.query('products >= 15').sort_values('unsold_share')
bad_brands = big_brands.query('unsold_share > 0.5')

fig, ax = plt.subplots(figsize=(12, max(4, len(big_brands) * 0.35)))
ax.barh(big_brands.index, big_brands['unsold_share'] * 100,
        color=np.where(big_brands['unsold_share'] > 0.5, ACCENT, MUTED))
ax.axvline(50, color='black', linestyle='--', linewidth=1)
ax.grid(axis='y', visible=False)
ax.set_xlim(0, 100)
ax.set(title='Доля ни разу не заказанных товаров (бренды от 15 товаров)', xlabel='%', ylabel='')
fig.tight_layout()
save_fig(fig, 'unsold_brands.png')
plt.show()

bad_brands.sort_values('unsold_share', ascending=False)

**Вывод:** у брендов **Dali, KEF, Marantz и Pioneer** больше половины ассортимента не продаётся —
их позиции в каталоге стоит пересмотреть.

## 5. Отчёт по менеджерам

Какую долю выручки и подтверждённых заказов за месяц обеспечил каждый менеджер.

In [ ]:
managers = (df_confirmed.groupby('manager', as_index=False)
            .agg(revenue=('revenue', 'sum'), orders=('order_id', 'nunique')))
managers['revenue_share'] = (managers['revenue'] / managers['revenue'].sum() * 100).round(2)
managers['orders_share'] = (managers['orders'] / managers['orders'].sum() * 100).round(2)
managers = managers.sort_values('revenue_share', ascending=True, ignore_index=True)

fig, ax = plt.subplots(figsize=(12, 7))
y = np.arange(len(managers))
h = 0.4
bars_rev = ax.barh(y + h / 2, managers['revenue_share'], height=h, color=MAIN, label='% выручки')
bars_ord = ax.barh(y - h / 2, managers['orders_share'], height=h, color=MUTED, label='% заказов')
ax.bar_label(bars_rev, fmt='%.1f', padding=3, fontsize=9)
ax.bar_label(bars_ord, fmt='%.1f', padding=3, fontsize=9)
ax.set_yticks(y, managers['manager'])
ax.grid(axis='y', visible=False)
ax.set_xlim(right=max(managers['revenue_share'].max(), managers['orders_share'].max()) * 1.12)
ax.set(title='Вклад менеджеров в выручку и заказы', xlabel='%', ylabel='')
ax.legend(frameon=False, loc='lower right')
fig.tight_layout()
save_fig(fig, 'managers_report.png')
plt.show()

managers.sort_values('revenue_share', ascending=False, ignore_index=True)

**Выводы**

- Лидер по доле выручки — Маргарита Камертонова, по доле заказов — Виктор Тромбонов.
- Явной «пятёрки лидеров» с большим отрывом нет: показатели снижаются постепенно.
- Наименьший вклад — у Аркадия Октавина и Сергея Контрабасова; с ними стоит обсудить итоги месяца.

## Итоги

| Область | Результат |
|---|---|
| Данные | Резервная выгрузка собрана в три таблицы и объединена в общий датасет с курсом доллара |
| Сезонность | Основной поток заказов — в будни; по выходным и 8 Марта заказов почти нет |
| Аномалия | Всплеск 14 марта вызван сбоем CRM 13 марта: заказы массово отменились и были оформлены повторно |
| Метрики | Доля отмен — 11%, средний чек — около 6,6 млн ₽ |
| Бренды | Лидер по выручке — JBL; у Dali, KEF, Marantz и Pioneer больше половины ассортимента не продаётся |
| Менеджеры | Вклад сильно различается; лидеры — Маргарита Камертонова и Виктор Тромбонов |

**Рекомендации**

- Проверить работу CRM и настроить мониторинг массовых отмен, чтобы вовремя замечать сбои.
- Пересмотреть ассортимент брендов с большой долей непродаваемых товаров.
- Разобрать результаты менеджеров с наименьшим вкладом и учитывать в оценке отменённые заказы.

**Идеи для развития:** проанализировать категории товаров, изучить причины отмен, построить когорты клиентов.